In this tutorial/guide, we will be constructing effective prompts before sending to the Anthropic API for further processing (this content is derived from Anthropic skilljar site)

Prompt Engineering is about taking a prompt and improving to get more reliable, higher-quality outputs.

This process involves iterative refinemenet. E.g. starts with a basic prompt, evaluate its performance then systematically applying engineering techniques to improve it.

The following ia the iterative improvement process:

1. Set a goal - define what your prompt needs to accomplish

2. Write an initial prompt - create a basic first attempt

3. Evaluate the prompt - test against your criteria/benchmark

4. Apply prompt engineering techniques to improve performance

5. Re-evaluate - verify that changes improved results.

We repeat steps 4 and 5 until we are satisfied with the performance.

# Setting up an Evaluation Pipeline

The following is an example to create an eval pipeline. We will create a prompt that generates one-day meal plans for athletes. The prompt needs to take account an athlete's height, weight, goals, and dietary restrictions, then produce a comprehensive meal plan.

In [ ]:
evaluator = PromptEvaluator(max_concurrent_tasks=5)

dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3
)

def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing
"""
)

We construct a prompt, feed the prompt to an LLM, and have a separate LLM to grade/check the results.

A good prompt must be:
1. Clear
- Use simple language
- State exactly what you want
- Lead with a straightforward statement of the AI model's task
2. Direct
- Use instructions, not questions
- Start with direct action verbs like Write, Create, or Generate

When your prompts include a lot of content, Claude or any AI models can sometimes struggle to understand which pieces of text belong together or what different sections are supposed to understand.

We can solve this by adding XML tags. This allow us to add structure and clarity to our prompts.

For example, we can structure a prompt with athlete info just like the following separating the info pertaining to the athlete from the instructions to Claude.

In [ ]:
<athlete_information>
- Height: 6'2"
- Weight: 180 lbs
- Goal: Build muscle
- Dietary restrictions: Vegetarian
</athlete_information>

Generate a meal plan based on the athlete information above.

Next, we could also provide examples in our prompts. This approach is known as "one-shot" or "multi-shot" prompting

- One-shot = provide single example to establish the pattern

- Multi-shot = provide multiple examples to cover different scenarios

Providing examples to Claude/AI model can help with:
- identifying tricky/corner/edge cases
- defining complex output formats like specific JSON structures
- showing the exact style/tone that we want
- demostrating how to handle ambigious inputs

For example, the improved prompt below includes context explaning why sarcasm should be treated carefully.

In [ ]:
"Great game tonight!" → "Positive"
"Oh yeah, I really needed a flight delay tonight! Excellent!" → "Negative"

Other than providing the examples, we can also explain why it is the case. For example we provide the output is good for the following input/output pair. This helps Claude to understand the reasoning behind the good responses, not just the format.

In [ ]:
<ideal_output>
[Your example output here]
</ideal_output>

This example is well-structured, provides detailed information
on food choices and quantities, and aligns with the athlete's
goals and restrictions.